# Tutorial 04: Simulations with parameter sweep

If we want to run simulations for a large number of parameter combinations, we can use the helper script `run_simulation_set.py` in the `utilities/simulation_helper` folder. This script allows us to run multiple simulations with different input parameters in an automated manner.

In this mode, we can choose the type of simulation we want to run (i.e., `simulate_population_dyn.py`, `simulate_population_magrot_det.py` or `simulate_population_full.py`) via the `--simulator_type` argument. In addition, the number of cores used to run the simulation in parallel is specified through `--processes`. Finally, we can choose between two sampling approaches to determine the relevant simulation parameters through the argument `--sampling_type`. 

Specifically, if `--sampling_type = grid`, a regular multi-dimensional grid is created, and we need to provide the parameters in a linear spacing format `--parameter [low] [high] [steps]`. For example,
```
python utilities/experiment_helpers/run_simulation_set.py --simulator_type simulate_population_dyn --save_dir output/sim_helper --vk_c 100.0 200.0 20 --sampling_type grid
```
This command will run the dynamical simulation only and generate a sweep of `20` uniformly spaced samples for the `vk_c` parameter in the range `[100.0, 200.0]`. As this parameter is related to the exponential kick model `km_exp`, the script will first check if this model is properly set in the `config_simulator.py` file. If not, an error will be thrown.

On the other hand, if `--sampling_type = random`, the parameters are sampled uniformly at random within the provided limits. In this case, we need to provide the parameter ranges in the format `--parameter [low] [high]` and specify the `--sampling_size` argument, which sets the number of values drawn from a uniform distribution for each parameter. For example,
```
python utilities/experiment_helpers/run_simulation_set.py --simulator_type simulate_population_dyn --save_dir output/sim_helper --vk_c 100.0 200.0 --sampling_type random --sampling_size 20
```
This will generate a sweep of `20` randomly drawn samples for the `vk_c` parameter in the range `[100.0, 200.0]`.

In the following, we list the parameters that can be swept with the `run_simulation_set.py` script.

For the dynamical evolution:

* `vk_c` for the exponential kick-velocity model `km_exp`;
* `sigma_k` for the Maxwell kick-velocity model `km_maxwell`;
* `h_c` for the Galactic scale height in the exponential model of birthplaces.

For the magneto-rotational evolution:

* `P_initial_mean` and `P_initial_sigma` for the birth spin periods in the `normal` distribution model;
* `P_initial_log10_mean` and `P_initial_log10_sigma` for the birth spin periods in the `log-normal` distribution model;
* `B_initial_log10_mean` and `B_initial_log10_sigma` for the birth magnetic fields in the log-normal distribution model;
* `a_late` for the power-law index describing the late-time decay of the magnetic field.

For the radio emission:

* `L_radio_log10_mean` and `epsilon_L` for the power-law model describing the pulsars' radio luminosity.

Finally, we can also sweep over more than one parameter. For example, let us assume that we want to simulate 20 populations of neutron stars varying the parameters `P_initial_log10_mean` in the range -1.5 to -0.3 and the parameter `B_initial_log10_mean` in the range 12 to 14 using the `simulate_population_magrot_det.py` script and a previously simulated dynamical database saved in `data/example_simulation_dyn`. We can then run:
```
python utilities/experiment_helpers/run_simulation_set.py --simulator_type simulate_population_magrot_det --save_dir output/sim_helper --dyn_data data/example_simulation_dyn --P_initial_log10_mean -1.5 -0.3 --B_initial_log10_mean 12 14 --sampling_type random --sampling_size 20
```
In this way, a population is simulated for each of the 20 pairs of random values of `P_initial_log10_mean` and `B_initial_log10_mean`. The sweep will generate a directory `output/sim_helper`, which will contain a folder for each simulation (parameter combination) named with an identifier, i.e., `000000`, `000001`, `000002` and so on.

If instead `--sampling_type grid`, we need to specify the command as follows:
```
python utilities/experiment_helpers/run_simulation_set.py --simulator_type simulate_population_magrot_det --save_dir output/sim_helper --dyn_data data/example_simulation_dyn --P_initial_log10_mean -1.5 -0.3 10 --B_initial_log10_mean 12 14 10 --sampling_type grid
```
This simulates a population for each combination of values of `P_initial_log10_mean` and `B_initial_log10_mean`, i.e., the above case corresponds to `10 x 10 = 100` simulations. Again, each simulation will be saved in a separate directory `000000`, `000001`, `000002` and so on.

In [ ]:
import argparse
import collections
import json
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pathlib
import sys

import utilities.plot_settings
from mlpoppyns.simulator.config_simulator import cfg
from utilities.experiment_helpers.run_simulation_set import main

WARNING: If you see a warning here, make sure to set the path to the repository in the simulator configuration file. Otherwise, the examples below will not run.

## Setting up and running the simulation

We first specify the output directory where the simulation results will be saved.

In [ ]:
output_dir = "output/sim_helper"

As an example, we will focus on simulating the magneto-rotational evolution plus detection for 20 random combinations of the two parameters `P_initial_log10_mean` and `B_initial_log10_mean`. To this end, we run the simulations by specifying the relevant arguments and executing the `main` function of the `utilities.experiment_helpers.run_simulation_set` module.

NOTE: Depending on your computational resources, this could take a couple of hours to run.

In [ ]:
helper_args = argparse.Namespace(
    simulator_type="simulate_population_magrot_det",
    dyn_data="../../data/example_simulation_dyn",
    save_dir=output_dir,
    sampling_type="random",
    sampling_size=20,
    sigma_k=None,
    vk_c=None,
    h_c=None,
    P_initial_mean=None,
    P_initial_sigma=None,
    P_initial_log10_mean=[-1.5, -0.3],
    P_initial_log10_sigma=None,
    B_initial_log10_mean=[12, 14],
    B_initial_log10_sigma=None,
    a_late=None,
    L_radio_log10_mean=None,
    epsilon_L=None,
    processes=4,
    B_initial_log10_mean_comp1=None,
    B_initial_log10_sigma_comp1=None,
    B_initial_log10_mean_comp2=None,
    B_initial_log10_sigma_comp2=None,
    B_initial_log10_weight_comp1=None,
    B_initial_log10_rise_mean=None,
    B_initial_log10_rise_sigma=None,
    B_initial_log10_decay_mean=None,
    B_initial_log10_decay_sigma=None,
    B_initial_log10_slope=None,
)
main(helper_args)

## Check the output for a couple of simulations

To check the output of the script, we will look at some of the prerun simulations, we have stored in `data/example_simulation_helper_magrot`.

In [ ]:
output_dir_1 = "../../data/example_simulation_helper_magrot/000000"
output_dir_2 = "../../data/example_simulation_helper_magrot/000001"

Let us look at the parameters for the very first simulation.

In [ ]:
config_filename_1 = pathlib.Path().joinpath(output_dir_1, "configuration.json")
with config_filename_1.open("rt") as handle:
    config_1 = json.load(handle, object_hook=collections.OrderedDict)

print("P_initial_log10_mean:", config_1["P_initial_log10_mean"])
print("B_initial_log10_mean:", config_1["B_initial_log10_mean"])

We now read in our compressed `.pkl` files for this simulation.

In [ ]:
data_PMPS_1 = pd.read_pickle(
    pathlib.Path().joinpath(output_dir_1, "survey_PMPS_results.pkl.gz"),
    compression="gzip",
)
data_PMPS_1.columns

In [ ]:
data_SMPS_1 = pd.read_pickle(
    pathlib.Path().joinpath(output_dir_1, "survey_SMPS_results.pkl.gz"),
    compression="gzip",
)
data_SMPS_1.columns

In [ ]:
data_HTRU_low_mid_1 = pd.read_pickle(
    pathlib.Path().joinpath(
        output_dir_1, "survey_HTRU_low_mid_results.pkl.gz"
    ),
    compression="gzip",
)
data_HTRU_low_mid_1.columns

Extracting the parameters.

In [ ]:
P_pk_sim_1 = data_PMPS_1["P"]["[s]"].to_numpy()
Pdot_pk_sim_1 = data_PMPS_1["P_dot"]["[s s^-1]"].to_numpy()
P_sw_sim_1 = data_SMPS_1["P"]["[s]"].to_numpy()
Pdot_sw_sim_1 = data_SMPS_1["P_dot"]["[s s^-1]"].to_numpy()
P_htru_sim_1 = data_HTRU_low_mid_1["P"]["[s]"].to_numpy()
Pdot_htru_sim_1 = data_HTRU_low_mid_1["P_dot"]["[s s^-1]"].to_numpy()

Plotting the simulation results for the first simulation in the $P-\dot{P}$ plane.

In [ ]:
fig, ax = plt.subplots(figsize=(15, 12))

ax.plot(
    P_pk_sim_1,
    Pdot_pk_sim_1,
    linestyle="None",
    marker="o",
    color="tab:red",
    markersize=6,
    alpha=1.0,
    rasterized=True,
    label=r"Simulated PMPS",
)
ax.plot(
    P_sw_sim_1,
    Pdot_sw_sim_1,
    linestyle="None",
    marker="o",
    color="tab:blue",
    markersize=6,
    alpha=1.0,
    rasterized=True,
    label=r"Simulated SMPS",
)
ax.plot(
    P_htru_sim_1,
    Pdot_htru_sim_1,
    linestyle="None",
    marker="o",
    fillstyle="none",
    color="tab:purple",
    markersize=6,
    alpha=1.0,
    rasterized=True,
    label=r"Simulated HTRU",
)

ax.set_xscale("log")
ax.set_yscale("log")
ax.set_xlim(1.0e-2, 30.0)
ax.set_ylim(1.0e-19, 1.0e-11)

plt.xlabel(r"$P$ [s]")
plt.ylabel(r"$\dot{P}$ [s/s]")
ax.legend(frameon=True, loc=2)

plt.show()

Looking at the parameters for the second simulation.

In [ ]:
config_filename_2 = pathlib.Path().joinpath(output_dir_2, "configuration.json")
with config_filename_2.open("rt") as handle:
    config_2 = json.load(handle, object_hook=collections.OrderedDict)

print("P_initial_log10_mean:", config_2["P_initial_log10_mean"])
print("B_initial_log10_mean:", config_2["B_initial_log10_mean"])

Reading in our compressed `.pkl` files for this simulation.

In [ ]:
data_PMPS_2 = pd.read_pickle(
    pathlib.Path().joinpath(output_dir_2, "survey_PMPS_results.pkl.gz"),
    compression="gzip",
)
data_PMPS_2.columns

In [ ]:
data_SMPS_2 = pd.read_pickle(
    pathlib.Path().joinpath(output_dir_2, "survey_SMPS_results.pkl.gz"),
    compression="gzip",
)
data_SMPS_2.columns

In [ ]:
data_HTRU_low_mid_2 = pd.read_pickle(
    pathlib.Path().joinpath(
        output_dir_2, "survey_HTRU_low_mid_results.pkl.gz"
    ),
    compression="gzip",
)
data_HTRU_low_mid_2.columns

Extracting the parameters.

In [ ]:
P_pk_sim_2 = data_PMPS_2["P"]["[s]"].to_numpy()
Pdot_pk_sim_2 = data_PMPS_2["P_dot"]["[s s^-1]"].to_numpy()
P_sw_sim_2 = data_SMPS_2["P"]["[s]"].to_numpy()
Pdot_sw_sim_2 = data_SMPS_2["P_dot"]["[s s^-1]"].to_numpy()
P_htru_sim_2 = data_HTRU_low_mid_2["P"]["[s]"].to_numpy()
Pdot_htru_sim_2 = data_HTRU_low_mid_2["P_dot"]["[s s^-1]"].to_numpy()

Plotting the simulation results for the second simulation in the $P-\dot{P}$ plane.

In [ ]:
fig, ax = plt.subplots(figsize=(15, 12))

ax.plot(
    P_pk_sim_2,
    Pdot_pk_sim_2,
    linestyle="None",
    marker="o",
    color="tab:red",
    markersize=6,
    alpha=1.0,
    rasterized=True,
    label=r"Simulated PMPS",
)
ax.plot(
    P_sw_sim_2,
    Pdot_sw_sim_2,
    linestyle="None",
    marker="o",
    color="tab:blue",
    markersize=6,
    alpha=1.0,
    rasterized=True,
    label=r"Simulated SMPS",
)
ax.plot(
    P_htru_sim_2,
    Pdot_htru_sim_2,
    linestyle="None",
    marker="o",
    fillstyle="none",
    color="tab:purple",
    markersize=6,
    alpha=1.0,
    rasterized=True,
    label=r"Simulated HTRU",
)

ax.set_xscale("log")
ax.set_yscale("log")
ax.set_xlim(1.0e-2, 30.0)
ax.set_ylim(1.0e-19, 1.0e-11)

plt.xlabel(r"$P$ [s]")
plt.ylabel(r"$\dot{P}$ [s/s]")
ax.legend(frameon=True, loc=2)

plt.show()